# Cross-evaluation pipeline — ResNet classifier ↔ GAN discriminator

This notebook is a **pipeline**: paste the paths at §1, run all, and it
produces two cross-evaluations that the project brief implicitly asks
for but that no other notebook in this project provides.

| Direction | Question it answers |
|---|---|
| **A. ResNet on GAN samples** | Does the deepfake classifier flag our own generated images as fake? Tracks how the classifier's verdict shifts as the GAN trains. |
| **B. GAN-D on the three deepfake datasets** | Does the GAN's discriminator (trained on wiki vs G's fakes) generalise to detecting *other* generators (SD-Inpainting, SD-1.5, InsightFace)? |

Both directions reuse one **generic evaluator** (§4) so adding a new
model/dataset is one line.

### What the user has to fill in

Only the **§1 paths cell** and (for direction B) one cell where the
`Discriminator` class is imported. Everything downstream picks up the
config automatically.

## §1 · Paths and configuration

All paths used by the pipeline live here. The defaults assume the
repository layout `outputs/` and `GAN/outputs/<run_name>/checkpoints/`.
Change them to point at any model or dataset and rerun the notebook.

In [ ]:
from pathlib import Path

# ─── classifier ─────────────────────────────────────────────────────
RESNET_CKPT     = Path("outputs/resnet_final_v6.pt")
RESNET_ARCH     = "resnet18"        # 'resnet18' or 'resnet34'
RESNET_DROPOUT  = 0.0               # match what was used at training

# ─── GAN discriminator ──────────────────────────────────────────────
GAN_CKPT        = Path("GAN/outputs/g10_fastgan_cropped/checkpoints/best_fid_checkpoint.pt")
GAN_IMG_SIZE    = 128               # most fastgan runs are 128; dcgan baselines were 64
GAN_NORMALISE   = "tanh"            # 'tanh' → normalise to [-1, 1]; 'imagenet' → ImageNet stats; 'none' → just ToTensor + Resize

# ─── data: the three real-deepfake datasets ─────────────────────────
from config import WIKI_DIR, INPAINTING_DIR, TEXT2IMG_DIR, INSIGHT_DIR
REAL_DEEPFAKE_DIRS = {
    "wiki":        WIKI_DIR,         # label 0 (real)
    "inpainting":  INPAINTING_DIR,
    "sd15":        TEXT2IMG_DIR,
    "insightface": INSIGHT_DIR,
}
EVAL_FOLDS = [80, 100]               # match resnetExploration.ipynb test split

# ─── data: GAN-generated samples ────────────────────────────────────
# A jsonl produced by a GAN training run — each line is {epoch, format, grid_b64}
GAN_SAMPLES_JSONL = Path("outputs/samples_03.jsonl")
GAN_SAMPLE_TILE   = None             # None → auto-detect from PNG; or pass int e.g. 64

# ─── output ─────────────────────────────────────────────────────────
OUT_DIR = Path("outputs") / "cross_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RESNET_CKPT       :", RESNET_CKPT,        "exists:", RESNET_CKPT.exists())
print("GAN_CKPT          :", GAN_CKPT,           "exists:", GAN_CKPT.exists())
print("GAN_SAMPLES_JSONL :", GAN_SAMPLES_JSONL,  "exists:", GAN_SAMPLES_JSONL.exists())
print("OUT_DIR           :", OUT_DIR)


## §2 · Imports and shared helpers

All scoring functions agree on a single convention:

> **score = predicted probability that the image is *fake*** (∈ [0, 1])

That means a real-trained ResNet returns its softmax `[:, 1]`, and a GAN
discriminator (which typically outputs *real-ness*) is **inverted** so
that higher scores mean "more likely synthetic". This single convention
makes histograms and scatter plots directly comparable across §5 and §6.

In [ ]:
import os, io, json, base64, copy, time
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from torchvision.models import resnet18, resnet34

from utils import DeepFakeDataset, eval_transform, set_global_seed

set_global_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## §3 · The generic evaluator

`run_scorer(scorer, dataset, name)` is the single function every direction
of the pipeline goes through. The contract is:

- `scorer` is a callable `(batch_tensor) → 1D probability array of "fake"`
- `dataset` is any PyTorch `Dataset` returning either `image_tensor` or
  `(image_tensor, label, ...)`
- returns a `pd.DataFrame` with one row per sample
  (`p_fake`, `label_if_known`)

Adding a new model or new dataset means writing one short scorer or one
short dataset and feeding it in. No other cell changes.

In [ ]:
@torch.no_grad()
def run_scorer(scorer, dataset, name, batch_size=128, max_samples=None):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=0, pin_memory=torch.cuda.is_available())
    p_fake, labels, n = [], [], 0
    for batch in loader:
        # Accept Dataset returning a bare tensor OR a (tensor, label, ...) tuple
        if isinstance(batch, (tuple, list)):
            imgs = batch[0]
            lbl  = batch[1] if len(batch) > 1 else None
        else:
            imgs, lbl = batch, None
        imgs = imgs.to(device, non_blocking=True)
        probs = scorer(imgs).detach().cpu().numpy()
        p_fake.extend(probs.tolist())
        if lbl is not None and hasattr(lbl, "numpy"):
            labels.extend(lbl.numpy().tolist())
        else:
            labels.extend([None] * len(imgs))
        n += len(imgs)
        if max_samples and n >= max_samples: break
    df = pd.DataFrame({"p_fake": p_fake[:n], "label": labels[:n]})
    df["source"] = name
    return df


def summarise(df):
    s = df["p_fake"]
    return {
        "n":              len(df),
        "mean_p_fake":    float(s.mean()),
        "median_p_fake":  float(s.median()),
        "pct_called_fake (p>0.5)": float((s > 0.5).mean() * 100),
        "p10":            float(s.quantile(0.10)),
        "p90":            float(s.quantile(0.90)),
    }


## §4 · Build the two scorers

### 4.1 · ResNet scorer

Loads the classifier checkpoint and wraps it so that calling the scorer
on a batch of ImageNet-normalised tensors returns `softmax(logits)[:, 1]`
(P(fake)).

In [ ]:
def build_resnet_scorer(ckpt_path, arch="resnet18", dropout=None):
    """Auto-detects the fc structure from the checkpoint keys.

    If dropout is None, inspects the state_dict:
      'fc.1.weight' present  -> model trained with dropout > 0
                                (fc = Sequential(Dropout, Linear))
      'fc.weight'   present  -> no dropout (fc = Linear).
    Pass dropout=<float> to force a specific value.
    """
    state = torch.load(ckpt_path, map_location=device)
    if dropout is None:
        if "fc.1.weight" in state:
            dropout = 0.5
            print("  detected fc = Sequential(Dropout, Linear) in checkpoint")
        elif "fc.weight" in state:
            dropout = 0.0
            print("  detected fc = Linear in checkpoint")
        else:
            fc_keys = sorted(k for k in state if k.startswith("fc"))
            raise KeyError(f"unknown fc structure; fc-keys in ckpt: {fc_keys}")

    if arch == "resnet18":
        model = resnet18(weights=None)
    elif arch == "resnet34":
        model = resnet34(weights=None)
    else:
        raise ValueError(arch)
    in_f = model.fc.in_features
    model.fc = (nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, 2))
                if dropout > 0 else nn.Linear(in_f, 2))
    model.load_state_dict(state)
    model.to(device).eval()

    def scorer(batch):
        return torch.softmax(model(batch), dim=1)[:, 1]
    scorer.__name__ = f"resnet({arch})"
    return scorer, model


# Pass dropout=None to auto-detect, or RESNET_DROPOUT to force a value.
resnet_scorer, resnet_model = build_resnet_scorer(
    RESNET_CKPT, arch=RESNET_ARCH, dropout=None
)
print("ResNet scorer ready:", resnet_scorer.__name__)


### 4.2 · GAN discriminator scorer  ✦ user customisation point ✦

The discriminator's architecture differs between GAN experiments
(DCGAN baseline vs FastGAN, 64 vs 128 image size, etc.). The cleanest
way is to:

1. **Try to load** the checkpoint with `weights_only=False` and recover
   the saved `Discriminator` instance directly (works when the training
   script saved the whole object, which `g9_fastgan` / `g10_fastgan_cropped`
   do — see `best_fid_checkpoint.pt`'s `'discriminator'` key).
2. **Fall back to a fresh import**: define / import `Discriminator` from
   wherever the training notebook defined it, instantiate, and load the
   state-dict. Edit the marked block below if option 1 fails.

The scorer **inverts** the output so that high = synthetic, matching
the ResNet convention.

In [ ]:
def build_gan_d_scorer(ckpt_path):
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    print("checkpoint keys:", list(ck.keys()) if hasattr(ck, "keys") else type(ck))

    # ── option 1: checkpoint stores the whole Discriminator object ──
    D = None
    if isinstance(ck, dict):
        for key in ("discriminator", "D", "netD", "discriminator_model"):
            if key in ck and hasattr(ck[key], "forward"):
                D = ck[key]
                print(f"loaded whole D from key '{key}'")
                break

    # ── option 2: checkpoint stores only a state_dict; user fills in ──
    if D is None:
        print("WARNING — checkpoint did not contain a full Discriminator "
              "object. Edit this block to import your D class and load "
              "its state_dict manually:")
        # ── EDIT THIS BLOCK ─────────────────────────────────────────
        # from GAN.your_module import Discriminator
        # D = Discriminator(ndf=64, nc=3)
        # sd = ck.get("D_state_dict") or ck.get("netD_state_dict") or ck
        # D.load_state_dict(sd)
        # ────────────────────────────────────────────────────────────
        raise RuntimeError(
            "GAN discriminator could not be auto-loaded — edit §4.2 "
            "fallback block to import your Discriminator class."
        )

    D = D.to(device).eval()

    @torch.no_grad()
    def scorer(batch):
        out = D(batch)
        if out.ndim > 1: out = out.view(out.size(0), -1).mean(dim=1)
        # D typically outputs raw logits or "realness" — convert to
        # P(fake) via sigmoid, then invert.
        return 1.0 - torch.sigmoid(out)
    scorer.__name__ = f"gan_D({ckpt_path.name})"
    return scorer, D


try:
    gan_d_scorer, gan_d_model = build_gan_d_scorer(GAN_CKPT)
    print("GAN-D scorer ready:", gan_d_scorer.__name__)
    GAN_D_OK = True
except Exception as e:
    print("GAN-D scorer not available:", e)
    print("→ §6 will be skipped. §5 still runs.")
    GAN_D_OK = False


## §5 · Data adapters

Two short Dataset classes feed the same `eval_transform` to whatever
images we have:

- `RealDeepfakeFolder` — wraps the existing `DeepFakeDataset` and only
  applies the test-fold range, so we're scoring on the same images the
  resnet test split uses.
- `GANGridSamples` — reads a `samples_*.jsonl` file, decodes each
  PNG grid, auto-detects the tile size, and yields one PIL image per
  sample. Each yielded sample carries a `(epoch, tile_row, tile_col)`
  identifier so the per-epoch story plot in §7 is trivial.

Both produce a tensor in the **transform of the scorer's choice** — see
the helper `build_dataset_for(scorer)` which picks the right transform.

In [ ]:
def resnet_transform():
    # ResNet was trained on raw ToTensor + Resize (see utils.resnetFormat).
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((224, 224)),
    ])

def gan_transform(size, mode):
    pipe = [transforms.Resize((size, size)), transforms.ToTensor()]
    if mode == "tanh":
        pipe.append(transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)))
    elif mode == "imagenet":
        pipe.append(transforms.Normalize((0.485, 0.456, 0.406),
                                         (0.229, 0.224, 0.225)))
    return transforms.Compose(pipe)


class GANGridSamples(Dataset):
    '''Reads samples_*.jsonl from a GAN training run.
    Each line: epoch, format, grid_b64 — grid is a tiled PNG of NxM
    sample images. We auto-detect the tile size from the PNG dimensions
    if tile_size is None.
    '''
    def __init__(self, jsonl_path, transform, tile_size=None, max_epochs=None):
        self.records = []
        self.transform = transform
        self.tile_size = tile_size
        with open(jsonl_path, encoding="utf-8") as f:
            for i, line in enumerate(f):
                obj = json.loads(line)
                self.records.append(obj)
                if max_epochs and len(self.records) >= max_epochs: break
        # Decode all grids up-front into individual tiles (small enough)
        self.tiles = []                      # list of (epoch, row, col, PIL)
        for obj in self.records:
            grid_bytes = base64.b64decode(obj["grid_b64"])
            grid = Image.open(io.BytesIO(grid_bytes)).convert("RGB")
            W, H = grid.size
            ts = self.tile_size or self._auto_tile(W, H)
            cols, rows = W // ts, H // ts
            for r in range(rows):
                for c in range(cols):
                    tile = grid.crop((c*ts, r*ts, (c+1)*ts, (r+1)*ts))
                    self.tiles.append((obj.get("epoch", -1), r, c, tile))
        print(f"GANGridSamples: {len(self.records)} grids → {len(self.tiles)} samples")

    @staticmethod
    def _auto_tile(W, H):
        # Try the four most common DCGAN/FastGAN tile sizes, in order
        for ts in (64, 128, 32, 96, 256):
            if W % ts == 0 and H % ts == 0:
                return ts
        # Fallback: assume an 8x8 grid
        return W // 8

    def __len__(self): return len(self.tiles)
    def __getitem__(self, idx):
        epoch, r, c, tile = self.tiles[idx]
        return self.transform(tile), 1, epoch    # label=1 because all GAN-generated → fake


## §6 · Direction A — ResNet classifier on everything

We feed the ResNet scorer four things, one after another:

1. wiki test images (label 0)
2. inpainting test images (label 1)
3. sd15 test images (label 1)
4. insightface test images (label 1)
5. GAN-generated samples from `GAN_SAMPLES_JSONL` (all label 1)

Five rows of stats, plus a histogram of P(fake) per source.

In [ ]:
results_A = []

# (1–4) the four real-deepfake datasets, test folds only
rs_transform = resnet_transform()
for name, dir_ in REAL_DEEPFAKE_DIRS.items():
    label = 0 if name == "wiki" else 1
    ds = DeepFakeDataset(dir_, label=label, transform=rs_transform,
                         range_folds=EVAL_FOLDS)
    df = run_scorer(resnet_scorer, ds, name=f"resnet|{name}")
    df["true_label"] = label
    results_A.append(df)
    s = summarise(df)
    print(f"resnet|{name:11s}  n={s['n']:>5}  mean P(fake)={s['mean_p_fake']:.3f}  "
          f"called-fake={s['pct_called_fake (p>0.5)']:.1f}%")

# (5) GAN samples
if GAN_SAMPLES_JSONL.exists():
    gan_ds = GANGridSamples(GAN_SAMPLES_JSONL, transform=rs_transform,
                            tile_size=GAN_SAMPLE_TILE)
    df_gan = run_scorer(resnet_scorer, gan_ds, name="resnet|gan_samples")
    df_gan["true_label"] = 1
    df_gan["epoch"]      = [t[0] for t in gan_ds.tiles[:len(df_gan)]]
    results_A.append(df_gan)
    s = summarise(df_gan)
    print(f"resnet|gan_samples  n={s['n']:>5}  mean P(fake)={s['mean_p_fake']:.3f}  "
          f"called-fake={s['pct_called_fake (p>0.5)']:.1f}%")
else:
    print(f"[skip] GAN samples not found: {GAN_SAMPLES_JSONL}")

df_A = pd.concat(results_A, ignore_index=True)
df_A.to_csv(OUT_DIR / "resnet_on_all.csv", index=False)


In [ ]:
# Histogram of P(fake) per source
fig, ax = plt.subplots(figsize=(11, 5))
palette = {"resnet|wiki": "steelblue",
           "resnet|inpainting": "darkorange",
           "resnet|sd15": "crimson",
           "resnet|insightface": "seagreen",
           "resnet|gan_samples": "purple"}
for src, sub in df_A.groupby("source"):
    ax.hist(sub["p_fake"], bins=40, alpha=0.45, density=True,
            label=f"{src}  (n={len(sub)})",
            color=palette.get(src, "gray"))
ax.axvline(0.5, color="black", linestyle=":", alpha=0.6, label="decision threshold")
ax.set_xlabel("P(fake) from ResNet"); ax.set_ylabel("density")
ax.set_title("Direction A — ResNet's P(fake) per dataset", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "A_resnet_pfake_per_source.png", dpi=120); plt.show()


In [ ]:
# Stats table — one row per source
rows = []
for src, sub in df_A.groupby("source"):
    rows.append({"source": src, **summarise(sub)})
df_stats_A = pd.DataFrame(rows).round(3)
df_stats_A.to_csv(OUT_DIR / "resnet_summary.csv", index=False)
df_stats_A


### §6.1 · How the ResNet's verdict on GAN samples evolves with training

If `GAN_SAMPLES_JSONL` contains samples across many epochs, this plot
shows the mean P(fake) the ResNet assigns at each GAN training epoch.
A *decreasing* line over GAN training means "our GAN is learning to
fool the classifier" — direct evidence of generator quality.

In [ ]:
if "epoch" in df_A.columns and df_A["epoch"].notna().any():
    gan_sub = df_A[df_A["source"] == "resnet|gan_samples"].copy()
    if "epoch" not in gan_sub.columns or gan_sub["epoch"].isna().all():
        # Re-attach epochs from the dataset
        gan_sub["epoch"] = [t[0] for t in gan_ds.tiles[:len(gan_sub)]]
    by_epoch = gan_sub.groupby("epoch")["p_fake"].agg(["mean", "std", "count"])
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(by_epoch.index, by_epoch["mean"], "-o", color="purple",
            label="mean P(fake)")
    ax.fill_between(by_epoch.index,
                    by_epoch["mean"] - by_epoch["std"],
                    by_epoch["mean"] + by_epoch["std"],
                    alpha=0.2, color="purple", label="±1 std")
    ax.axhline(0.5, color="black", linestyle=":", alpha=0.6)
    ax.set_xlabel("GAN training epoch"); ax.set_ylabel("ResNet P(fake)")
    ax.set_title("Does the ResNet see our GAN samples as 'less fake' as training progresses?",
                 fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "A_resnet_pfake_by_gan_epoch.png", dpi=120); plt.show()
else:
    print("[skip] no per-sample epoch information available.")


## §7 · Direction B — GAN discriminator on the three deepfake datasets

The discriminator was trained to distinguish *wiki real* from *its
own generator's fakes*. Does that learned signal generalise to detecting
**other** generators?

We score the same five datasets (4 real-deepfake + 1 GAN-self) and
plot P(fake) under the inverted-D convention. If the GAN-D fires
strongly only on its own samples and uniformly on wiki vs the other
three generators, that's a clear sign the D learned a generator-
specific fingerprint rather than a general fake-vs-real boundary.

In [ ]:
if not GAN_D_OK:
    print("Direction B skipped — see §4.2 for how to make GAN-D available.")
else:
    results_B = []
    gd_transform = gan_transform(GAN_IMG_SIZE, GAN_NORMALISE)
    for name, dir_ in REAL_DEEPFAKE_DIRS.items():
        label = 0 if name == "wiki" else 1
        ds = DeepFakeDataset(dir_, label=label, transform=gd_transform,
                             range_folds=EVAL_FOLDS)
        df = run_scorer(gan_d_scorer, ds, name=f"D|{name}")
        df["true_label"] = label
        results_B.append(df)
        s = summarise(df)
        print(f"D|{name:11s}  n={s['n']:>5}  mean P(fake)={s['mean_p_fake']:.3f}  "
              f"called-fake={s['pct_called_fake (p>0.5)']:.1f}%")

    if GAN_SAMPLES_JSONL.exists():
        gan_ds_for_D = GANGridSamples(GAN_SAMPLES_JSONL, transform=gd_transform,
                                       tile_size=GAN_SAMPLE_TILE)
        df_gan_D = run_scorer(gan_d_scorer, gan_ds_for_D, name="D|gan_samples")
        df_gan_D["true_label"] = 1
        results_B.append(df_gan_D)
        s = summarise(df_gan_D)
        print(f"D|gan_samples  n={s['n']:>5}  mean P(fake)={s['mean_p_fake']:.3f}  "
              f"called-fake={s['pct_called_fake (p>0.5)']:.1f}%")

    df_B = pd.concat(results_B, ignore_index=True)
    df_B.to_csv(OUT_DIR / "ganD_on_all.csv", index=False)


In [ ]:
if GAN_D_OK:
    fig, ax = plt.subplots(figsize=(11, 5))
    palette_B = {"D|wiki": "steelblue",
                 "D|inpainting": "darkorange",
                 "D|sd15": "crimson",
                 "D|insightface": "seagreen",
                 "D|gan_samples": "purple"}
    for src, sub in df_B.groupby("source"):
        ax.hist(sub["p_fake"], bins=40, alpha=0.45, density=True,
                label=f"{src}  (n={len(sub)})",
                color=palette_B.get(src, "gray"))
    ax.axvline(0.5, color="black", linestyle=":", alpha=0.6, label="decision threshold")
    ax.set_xlabel("P(fake) from GAN discriminator (1 - sigmoid(D))")
    ax.set_ylabel("density")
    ax.set_title("Direction B — GAN-D's P(fake) per dataset", fontweight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "B_ganD_pfake_per_source.png", dpi=120); plt.show()

    rows = []
    for src, sub in df_B.groupby("source"):
        rows.append({"source": src, **summarise(sub)})
    df_stats_B = pd.DataFrame(rows).round(3)
    df_stats_B.to_csv(OUT_DIR / "ganD_summary.csv", index=False)
    df_stats_B


## §8 · Do the ResNet and the GAN-D agree?

The most interesting plot in this notebook. For each test image, plot
the ResNet's P(fake) on the x-axis and the GAN-D's P(fake) on the
y-axis. Points coloured by ground-truth source.

- **Top-right quadrant** — both models say fake. Strong evidence.
- **Bottom-left** — both say real. Strong evidence.
- **Top-left / bottom-right** — the two models disagree. These are the
  most informative samples for the discussion section: if they
  systematically come from a specific generator, that generator has
  features the two detectors disagree on.

In [ ]:
if GAN_D_OK:
    # Need to align by source+label. Easiest: re-score each dataset by both
    # models on the same fixed-order DataLoader so row i is the same image.
    paired = []
    for name, dir_ in REAL_DEEPFAKE_DIRS.items():
        label = 0 if name == "wiki" else 1
        ds_r = DeepFakeDataset(dir_, label=label, transform=resnet_transform(),
                               range_folds=EVAL_FOLDS)
        ds_d = DeepFakeDataset(dir_, label=label, transform=gd_transform,
                               range_folds=EVAL_FOLDS)
        df_r = run_scorer(resnet_scorer, ds_r, name=name)
        df_d = run_scorer(gan_d_scorer, ds_d, name=name)
        paired.append(pd.DataFrame({
            "source":      name,
            "label":       label,
            "p_fake_resnet": df_r["p_fake"].values,
            "p_fake_ganD":   df_d["p_fake"].values,
        }))
    df_pair = pd.concat(paired, ignore_index=True)
    df_pair.to_csv(OUT_DIR / "pairwise_scores.csv", index=False)

    fig, ax = plt.subplots(figsize=(7, 7))
    for src, sub in df_pair.groupby("source"):
        ax.scatter(sub["p_fake_resnet"], sub["p_fake_ganD"],
                   s=8, alpha=0.4,
                   label=f"{src}  (n={len(sub)})",
                   color={"wiki":"steelblue","inpainting":"darkorange",
                          "sd15":"crimson","insightface":"seagreen"}.get(src, "gray"))
    ax.axhline(0.5, color="black", linestyle=":", alpha=0.4)
    ax.axvline(0.5, color="black", linestyle=":", alpha=0.4)
    ax.set_xlabel("ResNet P(fake)"); ax.set_ylabel("GAN-D P(fake)")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title("Per-image agreement between ResNet and GAN-D", fontweight="bold")
    ax.legend(loc="upper left", fontsize=9); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "C_scatter_resnet_vs_ganD.png", dpi=120); plt.show()


## §9 · Summary table — paste-into-slides

A single row per (scorer, dataset). The columns
`pct_called_fake (p>0.5)` and `mean_p_fake` are the two numbers to
quote on slides.

In [ ]:
parts = [df_stats_A.assign(scorer="resnet")]
if GAN_D_OK:
    parts.append(df_stats_B.assign(scorer="gan_D"))
df_summary = pd.concat(parts, ignore_index=True)
df_summary.to_csv(OUT_DIR / "cross_eval_summary.csv", index=False)
df_summary
